# FacturaFlash — Exploración de módulos

Notebook para probar cada módulo del backend de forma independiente.
Corre las celdas en orden o salta a la sección que quieras probar.

In [ ]:
import sys
from pathlib import Path

# Asegura que el root del proyecto esté en el path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

print(f'Root del proyecto: {ROOT}')

## 1. Lector de QR (`backend/qr_reader.py`)

In [ ]:
from PIL import Image
from backend.qr_reader import extract_qr_data, mock_qr_data
import json

# Prueba con datos mock
print('── Datos QR de demo ──')
print(json.dumps(mock_qr_data(), indent=2, ensure_ascii=False))

In [ ]:
# Prueba con una imagen real
# image_path = ROOT / 'data' / 'samples' / 'tu_factura.jpg'
# image = Image.open(image_path)
# qr_data = extract_qr_data(image)
# print(json.dumps(qr_data, indent=2, ensure_ascii=False))

## 2. OCR con PaddleOCR (`backend/ocr.py`)

In [ ]:
from backend.ocr import mock_ocr_text

print('── Texto OCR simulado ──')
print(mock_ocr_text())

In [ ]:
# Prueba OCR real (descarga modelos ~120 MB la primera vez)
# from backend.ocr import extract_text
# image_path = ROOT / 'data' / 'samples' / 'tu_factura.jpg'
# image = Image.open(image_path)
# text = extract_text(image)
# print(text)

## 3. Extractor con Claude (`backend/extractor.py`)

In [ ]:
from backend.extractor import extract_products_demo
import pandas as pd

products = extract_products_demo()
df = pd.DataFrame(products)
print(f'{len(products)} productos de demo:')
df

In [ ]:
# Extracción real con Claude (requiere ANTHROPIC_API_KEY)
# from backend.extractor import extract_products
# from backend.ocr import mock_ocr_text
#
# ocr_text = mock_ocr_text()
# products = extract_products(ocr_text)
# pd.DataFrame(products)

## 4. Exportación (`backend/exporter.py`)

In [ ]:
from backend.extractor import extract_products_demo
from backend.exporter import to_excel_bytes, to_csv_string
import pandas as pd

df = pd.DataFrame(extract_products_demo())

# Exportar a Excel
xlsx_bytes = to_excel_bytes(df)
output_path = ROOT / 'notebooks' / 'test_export.xlsx'
output_path.write_bytes(xlsx_bytes)
print(f'Excel guardado en: {output_path}')

# CSV
print('\nCSV preview:')
print(to_csv_string(df))

## 5. Pipeline completo (end-to-end con mocks)

In [ ]:
from backend.qr_reader import mock_qr_data
from backend.ocr import mock_ocr_text
from backend.extractor import extract_products_demo
from backend.exporter import to_excel_bytes
import pandas as pd

# Simular el flujo completo
qr = mock_qr_data()
ocr = mock_ocr_text()
products = extract_products_demo()

print('QR:',  qr['serie_numero'], '| Total:', qr['monto_total'])
print(f'OCR: {len(ocr.splitlines())} líneas')
df = pd.DataFrame(products)
print(f'Productos: {len(df)} ítems | Subtotal: S/ {df["subtotal"].sum():,.2f}')
df